In [3]:
# ============================================================
# SIH 2026 — PHASE 3D
# ML TRAINING ENVIRONMENT
# ============================================================

!pip -q install xgboost shap joblib

print("Phase 3D environment ready.")

Phase 3D environment ready.


In [4]:
# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/SIH - 2026"

print("Project directory:")
print(PROJECT_DIR)

import os

assert os.path.exists(PROJECT_DIR), "ERROR: SIH - 2026 project folder not found."

print("Project folder verified.")

Mounted at /content/drive
Project directory:
/content/drive/MyDrive/SIH - 2026
Project folder verified.


In [5]:
# ============================================================
# LOAD FROZEN ENVIRONMENTAL DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = (
    PROJECT_DIR +
    "/data/processed/meghalaya_environmental_features.csv"
)

df_pos = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df_pos.shape)

assert df_pos.shape == (1052, 43), (
    f"Unexpected dataset shape: {df_pos.shape}"
)

print("\nDataset integrity:")
print("Rows:", len(df_pos))
print("Columns:", len(df_pos))
print("Duplicate rows:", df_pos.duplicated().sum())

assert df_pos.duplicated().sum() == 0

print("\nFrozen positive inventory verified.")

Dataset loaded successfully.
Shape: (1052, 43)

Dataset integrity:
Rows: 1052
Columns: 1052
Duplicate rows: 0

Frozen positive inventory verified.


In [6]:
# ============================================================
# LOAD PSEUDO-ABSENCE CANDIDATES
# ============================================================

NEG_PATH = (
    PROJECT_DIR +
    "/data/phase3/pseudo_absence_candidates.csv"
)

df_neg = pd.read_csv(NEG_PATH)

print("Pseudo-absence dataset loaded.")
print("Shape:", df_neg.shape)

print("\nColumns:")
print(df_neg.columns.tolist())

print("\nSampling tiers:")
if "sampling_tier" in df_neg.columns:
    print(df_neg["sampling_tier"].value_counts())
else:
    print("sampling_tier column not found — inspecting available columns.")

Pseudo-absence dataset loaded.
Shape: (3156, 39)

Columns:
['pseudo_id', 'sample_ratio_tier', 'label', 'spatial_block_id', 'spatial_block_name', 'latitude', 'longitude', 'min_distance_to_landslide_m', 'elevation', 'slope', 'aspect', 'plan_curvature', 'profile_curvature', 'twi', 'spi', 'landcover_code', 'landcover_name', 'ndvi_mean', 'soil_clay_fraction', 'soil_sand_fraction', 'soil_bulk_density', 'soil_ph', 'lithology_major', 'lithology_code', 'distance_to_roads', 'distance_to_streams', 'event_date', 'event_year', 'temporal_quality', 'rainfall_event_day', 'ari_3', 'ari_7', 'ari_15', 'ari_30', 'max_1day_7d', 'max_3day_30d', 'rainy_days_7d', 'rainy_days_15d', 'rainy_days_30d']

Sampling tiers:
sampling_tier column not found — inspecting available columns.


In [7]:
# ============================================================
# PSEUDO-ABSENCE STRUCTURE CHECK
# ============================================================

print("Pseudo-absence rows:", len(df_neg))
print("Pseudo-absence columns:", len(df_neg.columns))

print("\nFirst 5 rows:")
display(df_neg.head())

print("\nMissing values:")
print(df_neg.isnull().sum()[df_neg.isnull().sum() > 0])

print("\nData types:")
print(df_neg.dtypes)

Pseudo-absence rows: 3156
Pseudo-absence columns: 39

First 5 rows:


,pseudo_id,sample_ratio_tier,label,spatial_block_id,spatial_block_name,latitude,longitude,min_distance_to_landslide_m,elevation,slope,...,rainfall_event_day,ari_3,ari_7,ari_15,ari_30,max_1day_7d,max_3day_30d,rainy_days_7d,rainy_days_15d,rainy_days_30d
0,NEG_0001,1:1,0,1,Garo Hills Block,25.545145,90.100808,9357.4,69.15,21.80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NEG_0002,1:1,0,2,West Khasi Block,25.507368,91.594325,3485.7,1370.62,26.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NEG_0003,1:1,0,4,Ri-Bhoi Block,25.891651,92.348974,19871.1,1079.00,29.19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NEG_0004,1:1,0,2,West Khasi Block,25.308639,91.452902,660.0,1138.34,19.54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NEG_0005,1:1,0,2,West Khasi Block,25.582522,91.460925,3815.2,1285.88,18.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values:
event_date            3156
event_year            3156
rainfall_event_day    3156
ari_3                 3156
ari_7                 3156
ari_15                3156
ari_30                3156
max_1day_7d           3156
max_3day_30d          3156
rainy_days_7d         3156
rainy_days_15d        3156
rainy_days_30d        3156
dtype: int64

Data types:
pseudo_id                       object
sample_ratio_tier               object
label                            int64
spatial_block_id                 int64
spatial_block_name              object
latitude                       float64
longitude                      float64
min_distance_to_landslide_m    float64
elevation                      float64
slope                          float64
aspect                         float64
plan_curvature                 float64
profile_curvature              float64
twi                            float64
spi                            float64
landcover_code                   int64
landcover

In [8]:
# ============================================================
# PHASE 3D — STEP 1
# VERIFY PSEUDO-ABSENCE SAMPLING TIERS
# ============================================================

print("Pseudo-absence sampling tiers:")
print(df_neg["sample_ratio_tier"].value_counts().sort_index())

print("\nExpected:")
print("1:1 -> 1052")
print("1:2 -> 2104")
print("1:3 -> 3156")

assert (df_neg["sample_ratio_tier"] == "1:1").sum() == 1052
assert (df_neg["sample_ratio_tier"] == "1:2").sum() == 2104
assert (df_neg["sample_ratio_tier"] == "1:3").sum() == 3156

print("\nPASS: All three sampling tiers verified.")

Pseudo-absence sampling tiers:
sample_ratio_tier
1:1    1052
1:2    1052
1:3    1052
Name: count, dtype: int64

Expected:
1:1 -> 1052
1:2 -> 2104
1:3 -> 3156


AssertionError: 

### Explanation of Current `sample_ratio_tier` Assignment

The `create_pseudo_absences.py` script, as reflected in the loaded `df_neg` DataFrame, assigns the `sample_ratio_tier` labels in a **mutually exclusive** manner. This means that out of the total 3156 pseudo-absence points generated, there are:

*   1052 points explicitly labeled `"1:1"`
*   1052 points explicitly labeled `"1:2"`
*   1052 points explicitly labeled `"1:3"`

Each pseudo-absence point belongs to *only one* of these `sample_ratio_tier` categories. The `value_counts()` output clearly shows these three distinct groups, each with 1052 entries.

### Discrepancy with Phase 3B Specification

The Phase 3B specification, however, implies a **cumulative selection** for these tiers:

*   The **1:1 ratio** should correspond to a dataset with 1052 pseudo-absence points.
*   The **1:2 ratio** should correspond to a dataset with 2104 pseudo-absence points (i.e., the 1052 from `"1:1"` _plus_ an additional 1052 points).
*   The **1:3 ratio** should correspond to a dataset with 3156 pseudo-absence points (i.e., the 2104 from `"1:2"` _plus_ an additional 1052 points).

The current assertions in the notebook (e.g., `assert (df_neg["sample_ratio_tier"] == "1:2").sum() == 2104`) are designed to check these *cumulative* counts by directly filtering on a single tier label. Because the `sample_ratio_tier` column is assigned in a mutually exclusive way, filtering for `"1:2"` only yields 1052 points, leading to the `AssertionError`.

In [ ]:
from pathlib import Path

script_path = Path("/content/drive/MyDrive/SIH - 2026/src/create_pseudo_absences.py")

print("Exists:", script_path.exists())
print("Path:", script_path)

if script_path.exists():
    print("\n" + "=" * 80)
    print("CREATE_PSEUDO_ABSENCES.PY")
    print("=" * 80)
    print(script_path.read_text(encoding="utf-8"))
else:
    print("ERROR: File not found.")